In [1]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scanpy as sc
import torch
import pandas as pd
from scipy.spatial.distance import cdist
import scipy.sparse as sp
from tqdm import tqdm
from tqdm import trange

In [4]:
adata_1 = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')
adata_2 = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')


In [2]:
from torch.utils.data import Dataset
import pandas as pd
import scanpy as sc
import numpy as np
import torch
import scipy.sparse as sp
from scipy.spatial.distance import cdist
from tqdm import trange

class BagsDataset(Dataset):
    def __init__(self, input_data, immune_cell, max_instances=None, radius=200, resolution='low'):
        self.immune_cell = immune_cell
        self.max_instances = max_instances
        self.radius = radius
        self.resolution = resolution
        if isinstance(input_data, str):
            self.bags = self.create_bags_from_csv(input_data)
        elif isinstance(input_data, sc.AnnData):
            self.bags = self.create_bags_from_adata(input_data)
        else:
            raise ValueError("input_data must be either a path to a CSV file or an AnnData object")

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = bag['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        core_idx = bag['core_idx']
        gene_names = bag['gene_names']
        return distances, gene_expression, label, core_idx, gene_names

    def create_bags_from_csv(self, csv_file):
        data = pd.read_csv(csv_file)
        adata_radius_list = []
        for _, row in data.iterrows():
            adata_path = row['adata']
            adata = sc.read_h5ad(adata_path)
            radius = row['radius'] if 'radius' in row and not pd.isna(row['radius']) else self.radius
            resolution = row['resolution'] if 'resolution' in row and not pd.isna(row['resolution']) else self.resolution
            adata_radius_list.append((adata, radius, resolution))
            print(f"Processing: adata={adata_path.split('/')[-1]}, radius={radius}, resolution={resolution}")
        return self.create_bags(adata_radius_list)

    def create_bags_from_adata(self, adata):
        adata_radius_list = [(adata, self.radius, self.resolution)]
        return self.create_bags(adata_radius_list)

    def create_bags(self, adata_radius_list):
        bags = {}
        bag_id = 0

        for adata, radius, resolution in adata_radius_list:
            spatial_coords_x = adata.obs['X'].astype(float)
            spatial_coords_y = adata.obs['Y'].astype(float)
            spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
            gene_expression = adata.X
            if self.immune_cell == 'tcell':
                labels = adata.obs['T'].values
            elif self.immune_cell == 'bcell':
                labels = adata.obs['B'].values
            else:
                raise ValueError("immune_cell must be either 'tcell' or 'bcell'")
            cell_types = adata.obs['cell_type'].values
            barcodes = adata.obs.index.values
            gene_names = adata.var_names.tolist()
            
            for i in trange(len(spatial_coords), desc=f"Creating Bags with radius {radius}", ncols=100, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"):
                if cell_types[i] == 0:
                    continue

                dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
                in_circle = np.where(dist_matrix_row <= radius)[0]
                in_circle = [idx for idx in in_circle if cell_types[idx] != 0]

                if resolution == 'high':
                    if cell_types[i] == 1:
                        in_circle.append(i)
                    else:
                        in_circle = [idx for idx in in_circle if idx != i]

                if len(in_circle) == 0:
                    continue

                if self.max_instances is not None and len(in_circle) > self.max_instances:
                    continue

                gene_data = gene_expression[in_circle]
                distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

                bags[bag_id] = {
                    'distances': distances,
                    'gene_expression': gene_data,
                    'label': labels[i],
                    'core_idx': i,
                    'gene_names': gene_names
                }

                bag_id += 1

        total_bags = len(bags)
        avg_instances_per_bag = sum(bags[i]['gene_expression'].shape[0] for i in bags) / total_bags if total_bags > 0 else 0
        print(f"Total bags created: {total_bags}")
        print(f"Average instances per bag: {avg_instances_per_bag:.0f}")

        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, labels, core_idxs, gene_names_list = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions_tensors = []
    for g in gene_expressions:
        if sp.issparse(g):
            gene_expressions_tensors.append(torch.tensor(g.todense(), dtype=torch.float32))
        else:
            gene_expressions_tensors.append(g.clone().detach().float())
    labels = torch.tensor(labels, dtype=torch.float32)
    core_idxs = torch.tensor(core_idxs, dtype=torch.long)
    gene_names = gene_names_list
    return distances, gene_expressions_tensors, labels, core_idxs, gene_names


In [7]:
dataset = BagsDataset(adata_1, immune_cell='tcell', max_instances=100, radius=200, resolution='low')


Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:01<00:00, 3525.17it/s]

Total bags created: 3858
Average instances per bag: 9


In [8]:
dataset = BagsDataset('./training.csv', immune_cell='tcell', max_instances=500)


Processing: adata=HumanLungCancerFFPE.h5ad, radius=200.0, resolution=low
Processing: adata=test.h5ad, radius=200, resolution=high


Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 6037.91it/s]

Total bags created: 7716
Average instances per bag: 8


In [108]:
dataset_1 = BagsDataset(adata_1, radius= 200, immune_cell='tcell',resolution='high')
dataset_2 = BagsDataset(adata_2, radius= 200, immune_cell='tcell',resolution='low')

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:01<00:00, 3855.13it/s]


Total bags created: 3858
Average instances per bag: 8


Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 4284.02it/s]

Total bags created: 3858
Average instances per bag: 9


In [109]:
dataset = dataset_1 + dataset_2

In [127]:
dataloader = DataLoader(dataset_1, batch_size=1, collate_fn=custom_collate_fn)

In [128]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))

In [130]:
current_genes


(['SAMD11',
  'NOC2L',
  'KLHL17',
  'PLEKHN1',
  'PERM1',
  'HES4',
  'ISG15',
  'AGRN',
  'RNF223',
  'C1orf159',
  'TTLL10',
  'TNFRSF18',
  'TNFRSF4',
  'SDF4',
  'B3GALT6',
  'C1QTNF12',
  'UBE2J2',
  'SCNN1D',
  'ACAP3',
  'PUSL1',
  'INTS11',
  'CPTP',
  'TAS1R3',
  'DVL1',
  'MXRA8',
  'AURKAIP1',
  'CCNL2',
  'ANKRD65',
  'TMEM88B',
  'VWA1',
  'ATAD3C',
  'ATAD3B',
  'ATAD3A',
  'TMEM240',
  'SSU72',
  'FNDC10',
  'MIB2',
  'MMP23B',
  'CDK11B',
  'CDK11A',
  'NADK',
  'GNB1',
  'CALML6',
  'TMEM52',
  'CFAP74',
  'GABRD',
  'PRKCZ',
  'FAAP20',
  'SKI',
  'RER1',
  'PEX10',
  'PLCH2',
  'PANK4',
  'HES5',
  'TNFRSF14',
  'PRXL2B',
  'MMEL1',
  'ACTRT2',
  'PRDM16',
  'ARHGEF16',
  'MEGF6',
  'TPRG1L',
  'WRAP73',
  'TP73',
  'CCDC27',
  'SMIM1',
  'LRRC47',
  'CEP104',
  'DFFB',
  'C1orf174',
  'AJAP1',
  'NPHP4',
  'KCNAB2',
  'CHD5',
  'RNF207',
  'ICMT',
  'GPR153',
  'ACOT7',
  'HES2',
  'ESPN',
  'TNFRSF25',
  'PLEKHG5',
  'NOL9',
  'TAS1R1',
  'ZBTB48',
  'KLHL21',
  '

In [134]:
current_genes_modified = list(current_genes[0])
len(current_genes_modified)

18085

In [132]:
adata_1.obs['cell_type']

AACACGTGCATCGCAC-1    0
AACACTTGGCAAGGAA-1    1
AACAGGATTCATAGTT-1    0
AACAGGTTATTGCACC-1    0
AACAGGTTCACCGAAG-1    0
                     ..
TGTTGGAACCTTCCGC-1    0
TGTTGGAACGAGGTCA-1    1
TGTTGGAAGCTCGGTA-1    0
TGTTGGATGGACTTCT-1    0
TGTTGGCCAGACCTAC-1    0
Name: cell_type, Length: 3858, dtype: category
Categories (2, object): ['0', '1']

In [44]:
len(adata_1.obs[adata_1.obs['cell_type'].astype(int) == 0])

2525

In [45]:
len(dataloader)

7716

In [46]:
distances, gene_expressions_tensors, labels, core_idxs, current_genes = next(iter(dataloader))

In [47]:
len(current_genes)

18085

In [48]:
dataset.cumulative_sizes

[3858, 7716]

In [121]:
first_bag['gene_expression'].shape

(4, 500)

In [122]:
dataloader.dataset.bags[2]['gene_expression']

array([[-0.6111422 , -0.37874946,  0.7928396 , ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ],
       [-0.6111422 , -0.37874946, -0.94990546, ..., -1.1006578 ,
        -1.058933  , -1.0542383 ]], dtype=float32)

In [123]:
dataloader.dataset.bags[2]['distances']

matrix([[ 0.      ],
        [16.00408 ],
        [16.004028],
        [15.994598]], dtype=float32)

In [124]:
dataloader.dataset.bags[0]['label']

0

In [125]:
dataset.__getitem__(2)

(tensor([[ 0.0000],
         [16.0041],
         [16.0040],
         [15.9946]]),
 tensor([[-0.6111, -0.3787,  0.7928,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542],
         [-0.6111, -0.3787, -0.9499,  ..., -1.1007, -1.0589, -1.0542]]),
 tensor(1.))

process the visium H5AD

In [3]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/test/HumanColorectalCancer/HumanColorectalCancer.h5ad')

In [7]:
adata.obs['t_gene_signature'].mean()

np.float32(-4.1684413e-07)

In [1]:
def predict(model, adata, binding_csv, device, radius=50, batch_size=1):
    model.eval()
    binding_aff = pd.read_csv(binding_csv, index_col=0)
    dataset = BagsDataset(adata, binding_aff, radius)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = []

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, affinity_data, _ in tepoch:
                tepoch.set_description("Predicting")
                
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                affinity_data = [a.to(device) for a in affinity_data]

                instance_outputs = []
                for dist, gene_exp, aff_data in zip(distances, gene_expressions, affinity_data):
                    output = model(dist, gene_exp, aff_data)
                    instance_outputs.append(output)
                
                instance_outputs = torch.cat(instance_outputs, dim=0)
                bag_output = torch.mean(instance_outputs, dim=0, keepdim=True)
                predictions.append(bag_output.cpu().numpy().flatten())

    predictions = np.array(predictions)
    adata.obs['tcr_predict'] = predictions
    return adata

In [2]:
adata = predict(model, adata, binding_affinity)

NameError: name 'model' is not defined

In [ ]:


#can run with warnigns
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
import pandas as pd
from scipy.spatial.distance import cdist
import scipy.sparse as sp

class BagsDataset(Dataset):
    def __init__(self, adata, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = bag['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        return distances, gene_expression, label

    def create_bags(self, adata, radius, output_csv):
        spatial_coords_x = adata.obs['X'].astype(float)
        spatial_coords_y = adata.obs['Y'].astype(float)
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        cell_types = adata.obs['cell_type'].values
        barcodes = adata.obs.index.values

        bags = {}
        csv_data = []
        filtered_count = 0
        no_neighbors_count = 0
        bag_id = 0  # Initialize bag_id to ensure continuous IDs starting from 0

        for i in range(len(spatial_coords)):
            if cell_types[i] == 0:
                continue  # Skip if the cell type is 0

            filtered_count += 1
            dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
            in_circle = np.where(dist_matrix_row <= radius)[0]
            in_circle = [idx for idx in in_circle if cell_types[idx] != 0]  # Filter based on cell type
            if len(in_circle) == 0:
                no_neighbors_count += 1
                continue  # Skip if no instances meet the criteria

            gene_data = gene_expression[in_circle]
            distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

            bags[bag_id] = {
                'distances': distances,
                'gene_expression': gene_data,
                'label': labels[i]
            }

            bag_barcodes = barcodes[in_circle]
            for barcode in bag_barcodes:
                csv_data.append([bag_id, barcode, labels[i]])

            print(f"Bag {bag_id} has {gene_data.shape[0]} instances")
            bag_id += 1  # Increment bag_id for the next bag

        print(f"Total points filtered by cell type: {filtered_count}")
        print(f"Total bags created: {len(bags)}")
        print(f"Total points with no neighbors: {no_neighbors_count}")

        df = pd.DataFrame(csv_data, columns=['bag_id', 'barcode', 'label'])
        df.to_csv(output_csv, index=False)
        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, labels = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions_tensors = []
    for g in gene_expressions:
        if sp.issparse(g):
            gene_expressions_tensors.append(torch.tensor(g.todense(), dtype=torch.float32))
        else:
            gene_expressions_tensors.append(torch.tensor(g, dtype=torch.float32))
    labels = torch.tensor(labels, dtype=torch.float32)
    return distances, gene_expressions_tensors, labels